In [1]:
from pyspark.sql.functions import col, sum, count

StatementMeta(, 16a3d63d-c343-42fb-92ac-7334960ffbd6, 3, Finished, Available, Finished, False)

In [2]:
# Read sales order data from the Lakehouse
df = spark.read.table("SalesOrders")
display(df.limit(10))

# Save raw data to the Bronze layer
df.write.format("delta").mode("overwrite").saveAsTable("Bronze_SalesOrders")

# Validate Bronze data
bronze_df = spark.read.table("Bronze_SalesOrders")
print("Total rows:", bronze_df.count())
print("Distinct sales orders:", bronze_df.select("SalesOrderID").distinct().count())

StatementMeta(, 16a3d63d-c343-42fb-92ac-7334960ffbd6, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9ef07d2b-9fb9-406a-b1e3-e3bfc1fac8bf)

Total rows: 100
Distinct sales orders: 100


In [3]:
# Clean Bronze data by filtering out records with missing key fields
silver_df = (
    bronze_df
    .filter(
        col("SalesOrderID").isNotNull() &
        col("OrderDate").isNotNull() &
        col("CustomerID").isNotNull()
    )
)

# Save cleaned data to the Silver layer
silver_df.write.format("delta").mode("overwrite").saveAsTable("Silver_SalesOrders")
print("Silver rows:", spark.read.table("Silver_SalesOrders").count())

StatementMeta(, 16a3d63d-c343-42fb-92ac-7334960ffbd6, 5, Finished, Available, Finished, False)

Silver rows: 100


In [4]:
# Aggregate revenue and order counts by sales territory
silver_df = spark.read.table("Silver_SalesOrders")

gold_df = (
    silver_df
    .groupBy("TerritoryID")
    .agg(
        sum("TotalDue").alias("TotalRevenue"),
        count("SalesOrderID").alias("OrderCount")
    )
    .orderBy("TerritoryID")
)

# Save aggregated results to the Gold layer
gold_df.write.format("delta").mode("overwrite").saveAsTable("Gold_SalesByTerritory")

# Validate the Gold layer
display(gold_df)
print("Total orders in Gold:", gold_df.agg({"OrderCount": "sum"}).collect()[0][0])
print("Number of territories:", gold_df.count())

StatementMeta(, 16a3d63d-c343-42fb-92ac-7334960ffbd6, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dc056c42-68f8-4120-a27c-36aeaf70ea60)

Total orders in Gold: 100
Number of territories: 10


In [5]:
gold_df = spark.read.table("Gold_SalesByTerritory")

print("Total orders in Gold:", gold_df.agg({"OrderCount": "sum"}).collect()[0][0])
print("Number of territories:", gold_df.count())

StatementMeta(, 16a3d63d-c343-42fb-92ac-7334960ffbd6, 7, Finished, Available, Finished, False)

Total orders in Gold: 100
Number of territories: 10


In [1]:
# Incremental loading: inspect the existing Bronze table

from pyspark.sql.functions import count, countDistinct, max

bronze_df = spark.read.table("Bronze_SalesOrders")

print("Total rows:", bronze_df.count())
print("Unique order IDs:", bronze_df.select("SalesOrderID").distinct().count())
print("Latest order ID:", bronze_df.agg(max("SalesOrderID")).collect()[0][0])

StatementMeta(, b194d653-54fe-4e5d-87a9-5f45575ec943, 3, Finished, Available, Finished, False)

Total rows: 100
Unique order IDs: 100
Latest order ID: 43758


In [2]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType,
    DoubleType, StringType
)

# Create a small, separate dataset for incremental-loading practice
schema = StructType([
    StructField("SalesOrderID", IntegerType(), False),
    StructField("CustomerID", IntegerType(), True),
    StructField("TerritoryID", IntegerType(), True),
    StructField("TotalDue", DoubleType(), True),
    StructField("ModifiedDate", StringType(), True)
])

initial_orders = [
    (900001, 1001, 1, 150.00, "2026-10-26 10:00:00"),
    (900002, 1002, 2, 250.00, "2026-10-26 10:05:00"),
    (900003, 1003, 1, 300.00, "2026-10-26 10:10:00")
]

incremental_df = spark.createDataFrame(initial_orders, schema)

incremental_df.write.format("delta").mode("overwrite").saveAsTable(
    "IncrementalOrders_Demo"
)

print("Initial test batch created.")
print("Rows:", spark.read.table("IncrementalOrders_Demo").count())

StatementMeta(, b194d653-54fe-4e5d-87a9-5f45575ec943, 4, Finished, Available, Finished, False)

Initial test batch created.
Rows: 3


In [3]:
from pyspark.sql.functions import col

# Simulate two new orders arriving
new_orders = [
    (900004, 1004, 2, 175.00, "2026-10-27 09:00:00"),
    (900005, 1005, 3, 425.00, "2026-10-27 09:05:00")
]

new_batch = spark.createDataFrame(new_orders, schema)

# Append the new records to the existing demo table
new_batch.write.format("delta").mode("append").saveAsTable(
    "IncrementalOrders_Demo"
)

# Verify the result
demo_df = spark.read.table("IncrementalOrders_Demo")

print("Total rows:", demo_df.count())
print("Distinct order IDs:", demo_df.select("SalesOrderID").distinct().count())

StatementMeta(, b194d653-54fe-4e5d-87a9-5f45575ec943, 5, Finished, Available, Finished, False)

Total rows: 5
Distinct order IDs: 5


In [4]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# Simulate an update to existing order 900002
updated_order = [
    (900002, 1002, 2, 200.00, "2026-10-27 10:00:00")
]

updates_df = spark.createDataFrame(updated_order, schema)

# Update the matching order; insert only if it does not exist
delta_table = DeltaTable.forName(spark, "IncrementalOrders_Demo")

(
    delta_table.alias("target")
    .merge(
        updates_df.alias("source"),
        "target.SalesOrderID = source.SalesOrderID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

# Validate the result
demo_df = spark.read.table("IncrementalOrders_Demo")

print("Total rows:", demo_df.count())
print(
    "Order 900002 total:",
    demo_df.filter(col("SalesOrderID") == 900002)
           .select("TotalDue").first()[0]
)

StatementMeta(, b194d653-54fe-4e5d-87a9-5f45575ec943, 6, Finished, Available, Finished, False)

Total rows: 5
Order 900002 total: 200.0


In [5]:
from pyspark.sql.functions import col

demo_df = spark.read.table("IncrementalOrders_Demo")

print("Total orders:", demo_df.count())
print("Unique order IDs:", demo_df.select("SalesOrderID").distinct().count())
print("Updated order 900002:")

demo_df.filter(col("SalesOrderID") == 900002).show()

StatementMeta(, b194d653-54fe-4e5d-87a9-5f45575ec943, 7, Finished, Available, Finished, False)

Total orders: 5
Unique order IDs: 5
Updated order 900002:
+------------+----------+-----------+--------+-------------------+
|SalesOrderID|CustomerID|TerritoryID|TotalDue|       ModifiedDate|
+------------+----------+-----------+--------+-------------------+
|      900002|      1002|          2|   200.0|2026-10-27 10:00:00|
+------------+----------+-----------+--------+-------------------+

